# Catalog Helpers Notebook

This notebook provides reusable utilities for schema and table management in the CDC pipeline.

## Functions
- `ensure_schema_exists(catalog, schema)` - Create schema if not exists
- `get_existing_schema(spark, table_fqn)` - Get existing Delta table schema
- `get_existing_columns(spark, table_fqn)` - Get set of column names
- `create_silver_table_rental(spark, table_fqn)` - Create rental Silver table
- `create_silver_table_film(spark, table_fqn)` - Create film Silver table
- `create_silver_table_payment(spark, table_fqn)` - Create payment Silver table
- `build_merge_clauses(existing_columns, core_fields, ...)` - Build MERGE update/insert clauses
- `execute_merge(spark, delta_table, df, update_set, insert_values, ...)` - Execute MERGE

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, DecimalType


def ensure_schema_exists(catalog: str, schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema {catalog}.{schema} is ready")


def get_existing_schema(spark, table_fqn: str) -> StructType | None:
    try:
        return spark.table(table_fqn).schema
    except Exception:
        return None


def get_existing_columns(spark, table_fqn: str) -> set:
    try:
        return set(spark.table(table_fqn).columns)
    except Exception:
        return set()


def create_silver_table_rental(spark, table_fqn: str) -> None:
    """Create the Silver rental table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            rental_id       INT         NOT NULL,
            rental_date     TIMESTAMP,
            inventory_id    INT,
            customer_id     INT,
            return_date     TIMESTAMP,
            staff_id        INT,
            last_update     TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP  NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver rental table {table_fqn} is ready")


def create_silver_table_film(spark, table_fqn: str) -> None:
    """Create the Silver film table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            film_id          INT            NOT NULL,
            title            STRING         NOT NULL,
            description      STRING,
            release_year     INT,
            language_id      INT,
            rental_duration  INT,
            rental_rate      DECIMAL(4,2),
            length           INT,
            replacement_cost DECIMAL(5,2),
            rating           STRING,
            last_update      TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP      NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver film table {table_fqn} is ready")


def create_silver_table_payment(spark, table_fqn: str) -> None:
    """Create the Silver payment table for dvdrental CDC upserts."""
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            payment_id      INT            NOT NULL,
            customer_id     INT,
            staff_id        INT,
            rental_id       INT,
            amount          DECIMAL(5,2),
            payment_date    TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt  TIMESTAMP     NOT NULL
        ) USING DELTA
        """
    )
    print(f"Silver payment table {table_fqn} is ready")


def build_merge_clauses(
    existing_columns: set,
    core_fields: list,
    include_inserted_dt: bool = True,
    include_updated_dt: bool = True
) -> tuple[dict, dict]:
    """
    Build dynamic MERGE update and insert clauses based on existing schema.
    The first element of core_fields is treated as the primary key (excluded from update_set).
    """
    pk = core_fields[0] if core_fields else None
    update_set = {}
    insert_values = {}

    for field in core_fields:
        if field in existing_columns:
            if field != pk:
                update_set[field] = f"s.{field}"
            insert_values[field] = f"s.{field}"

    if include_inserted_dt and "last_inserted_dt" in existing_columns:
        update_set["last_inserted_dt"] = "COALESCE(s.event_time, t.last_inserted_dt)"
        insert_values["last_inserted_dt"] = "s.event_time"

    if include_updated_dt and "last_updated_dt" in existing_columns:
        update_set["last_updated_dt"] = "s.event_time"
        insert_values["last_updated_dt"] = "s.event_time"

    return update_set, insert_values


def execute_merge(
    spark,
    delta_table: DeltaTable,
    latest_df,
    update_set: dict,
    insert_values: dict,
    join_condition: str = "t.id = s.id",
    delete_condition: str = "s.op = 'd'",
    update_condition: str = "s.op <> 'd'"
) -> None:
    """Execute a MERGE operation with dynamic clauses."""
    merge_builder = (
        delta_table.alias("t")
        .merge(latest_df.alias("s"), join_condition)
    )

    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(
            condition=update_condition,
            set=update_set
        )

    merge_builder = merge_builder.whenMatchedDelete(condition=delete_condition)

    if insert_values:
        merge_builder = merge_builder.whenNotMatchedInsert(
            condition=update_condition,
            values=insert_values
        )

    merge_builder.execute()

## Usage Example

```python
%run ./helpers/NB_catalog_helpers

ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
create_silver_table_rental(spark, silver_table_fqn)

existing_columns = get_existing_columns(spark, silver_table_fqn)
update_set, insert_values = build_merge_clauses(existing_columns, core_fields)
execute_merge(spark, delta_table, latest, update_set, insert_values, join_condition='t.rental_id = s.rental_id')
```